In [1]:
import pandas as pd

In [2]:
import os
import sys

# Set the working directory to the project root (RepClassifier)
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
os.chdir(project_root)

# Optionally, add the project root to the Python path
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Working directory set to: {os.getcwd()}")

Working directory set to: /Users/alexdominguez/Documents/GitHub/TFM/RepClassifier


In [3]:
# Set the dataset path to the parent directory of the current script
dataset_path = os.path.join(project_root, "../DATASETS/")

new_moonprot_3 = os.path.join(dataset_path, "MoonProt3forAna21925.csv")
seq_moonprot_3 = os.path.join(dataset_path, "MoonProt3_sequences.csv")
go_moonprot_3 = os.path.join(dataset_path, "MoonProt3_go_anotations.csv")

# Check if the files exist  and print their paths
def check_file(file_path):
    if os.path.exists(file_path):
        print(f"File exists: {file_path}")
    else:
        print(f"File does not exist: {file_path}")

check_file(new_moonprot_3)
check_file(seq_moonprot_3)
check_file(go_moonprot_3)

File exists: /Users/alexdominguez/Documents/GitHub/TFM/RepClassifier/../DATASETS/MoonProt3forAna21925.csv
File exists: /Users/alexdominguez/Documents/GitHub/TFM/RepClassifier/../DATASETS/MoonProt3_sequences.csv
File exists: /Users/alexdominguez/Documents/GitHub/TFM/RepClassifier/../DATASETS/MoonProt3_go_anotations.csv


In [4]:
# Ensure the file exists
check_file(new_moonprot_3)

# Load the dataset, skipping the first row and using the second row as the header
new_moonprot_3 = pd.read_csv(new_moonprot_3, sep=";", header=2)

# Verify the columns in the dataset
print("MoonProt 3 dataset columns:")
print(new_moonprot_3.columns)

File exists: /Users/alexdominguez/Documents/GitHub/TFM/RepClassifier/../DATASETS/MoonProt3forAna21925.csv
MoonProt 3 dataset columns:
Index(['id', 'title', 'names', 'uniprot', 'uniprot_id', 'go_terms',
       'organisms', 'sequence_length', 'quaternary_structure',
       'fasta_sequence', 'pdb_id', 'SCOP', 'CATH', 'TM_helix_prediction',
       'DisProt', 'Predicted_disorder', 'OMIM_ID', 'one_function',
       'one_references', 'one_pmid_references', 'one_ec_number',
       'one_location_functional', 'one_cellular_location', 'one_comments',
       'two_function', 'two_references', 'two_pmid_references',
       'two_ec_number', 'two_location_functional', 'two_cellular_location',
       'two_comments', 'Three_function', 'three_references',
       'three_pmid_references', 'three_ec_number', 'three_location_functional',
       'three_cellular_location', 'three_comments', 'protein_seq'],
      dtype='object')


In [5]:
# Load MoonProt 3 sequences dataset with explicit headers
sequence_headers = [
    "UniProt IDs", "Reviewed", "Sequence Length", "Validate Length", "Amino Acid Sequence"
]
moonprot_3_seq = pd.read_csv(seq_moonprot_3, sep=",", names=sequence_headers, header=0)

# Load MoonProt 3 GO annotations dataset with explicit headers
go_annotation_headers = [
    "UniProt IDs", "GO Annotation", "GO Evidence", "GoTerm", "GO Category"
]
moonprot_3_go = pd.read_csv(go_moonprot_3, sep=",", names=go_annotation_headers, header=0)

# Display the first few rows to verify
print("MoonProt 3 sequences dataset columns:")
print(moonprot_3_seq.columns)
print("MoonProt 3 GO annotations dataset columns:")
print(moonprot_3_go.columns)

MoonProt 3 sequences dataset columns:
Index(['UniProt IDs', 'Reviewed', 'Sequence Length', 'Validate Length',
       'Amino Acid Sequence'],
      dtype='object')
MoonProt 3 GO annotations dataset columns:
Index(['UniProt IDs', 'GO Annotation', 'GO Evidence', 'GoTerm', 'GO Category'], dtype='object')


In [6]:
new_uniprot_ids = new_moonprot_3["uniprot_id"].unique()
print(f"The dataset contains {len(new_moonprot_3)} entries.")
print(f"Number of unique UniProt IDs: {len(new_uniprot_ids)}")

print(f"Unique UniProt IDs from the sequences dataset: {len(moonprot_3_seq['UniProt IDs'].unique())}")
print(f"Unique UniProt IDs from the GO annotations dataset: {len(moonprot_3_go['UniProt IDs'].unique())}")

The dataset contains 473 entries.
Number of unique UniProt IDs: 458
Unique UniProt IDs from the sequences dataset: 463
Unique UniProt IDs from the GO annotations dataset: 446


Okey, we see a discrepancy between the datasets, we're gonna go futher comparing the uniprot_ids, the go_terms and the sequences lengths

In [7]:
# Select only the desired columns
filtered_moonprot_3 = new_moonprot_3[['uniprot_id', 'go_terms', 'sequence_length']]

# Display the first few rows to verify
print(filtered_moonprot_3.head())

  uniprot_id                                           go_terms  \
0     Q9CW03  GO:0006275 regulation of DNA replication\r\r\n...   
1     Q43155  GO:0006537 glutamate biosynthetic process \r\r...   
2     P30041  GO:0006629 lipid metabolic process\r\n\r\nGO:0...   
3     P06745  GO:0001525 angiogenesis\r\n\r\nGO:0001701 in u...   
4     P10809  GO:0002368  B cell cytokine production  \r\n\r...   

  sequence_length  
0            1217  
1            1517  
2             224  
3             558  
4             NaN  


In [12]:
# Extract unique UniProt IDs from each dataset
seq_uniprot_ids = set(moonprot_3_seq["UniProt IDs"].unique())
go_uniprot_ids = set(moonprot_3_go["UniProt IDs"].unique())
filtered_uniprot_ids = set(filtered_moonprot_3["uniprot_id"].unique())

# Combine all unique UniProt IDs from the three datasets
all_uniprot_ids = seq_uniprot_ids.union(go_uniprot_ids).union(filtered_uniprot_ids)

# Create a list to store the results
comparison_results = []

# Check for each UniProt ID in all datasets
for uniprot_id in all_uniprot_ids:
    comparison_results.append({
        "UniProt ID": uniprot_id,
        "In Sequences Dataset": uniprot_id in seq_uniprot_ids,
        "In GO Annotations Dataset": uniprot_id in go_uniprot_ids,
        "In Filtered Dataset": uniprot_id in filtered_uniprot_ids
    })

# Convert the results to a DataFrame for better visualization
comparison_df = pd.DataFrame(comparison_results)

# Generate a summary
total_ids = len(all_uniprot_ids)
in_sequences_only = len(seq_uniprot_ids - go_uniprot_ids - filtered_uniprot_ids)
in_go_only = len(go_uniprot_ids - seq_uniprot_ids - filtered_uniprot_ids)
in_filtered_only = len(filtered_uniprot_ids - seq_uniprot_ids - go_uniprot_ids)
in_all_datasets = len(seq_uniprot_ids & go_uniprot_ids & filtered_uniprot_ids)

# Print the summary
print("\nSummary:")
print(f"Total unique UniProt IDs: {total_ids}")
print(f"UniProt IDs only in Sequences Dataset: {in_sequences_only}")
print(f"UniProt IDs only in GO Annotations Dataset: {in_go_only}")
print(f"UniProt IDs only in Filtered Dataset: {in_filtered_only}")
print(f"UniProt IDs present in all datasets: {in_all_datasets}")

# Optionally, save the comparison results to a CSV file
comparison_df.to_csv("uniprot_id_comparison.csv", index=False)


Summary:
Total unique UniProt IDs: 478
UniProt IDs only in Sequences Dataset: 0
UniProt IDs only in GO Annotations Dataset: 0
UniProt IDs only in Filtered Dataset: 15
UniProt IDs present in all datasets: 426


In [29]:
# Ensure the sequence length columns are numeric
moonprot_3_seq['Sequence Length'] = pd.to_numeric(moonprot_3_seq['Sequence Length'], errors='coerce')

# Create a copy of the filtered DataFrame to avoid the warning
filtered_moonprot_3 = filtered_moonprot_3.copy()
filtered_moonprot_3['sequence_length'] = pd.to_numeric(filtered_moonprot_3['sequence_length'], errors='coerce')

# Merge the two datasets on UniProt IDs
merged_lengths = pd.merge(
    moonprot_3_seq[['UniProt IDs', 'Sequence Length']],  # Select relevant columns from sequences dataset
    filtered_moonprot_3[['uniprot_id', 'sequence_length']],  # Select relevant columns from filtered dataset
    left_on='UniProt IDs', right_on='uniprot_id', how='outer'
)

# Rename columns for clarity
merged_lengths.rename(columns={
    'Sequence Length': 'Sequence Length (moonprot_3_seq)',
    'sequence_length': 'Sequence Length (filtered_moonprot_3)'
}, inplace=True)

# Calculate the difference in sequence lengths
merged_lengths['Length Difference'] = (
    merged_lengths['Sequence Length (moonprot_3_seq)'] - merged_lengths['Sequence Length (filtered_moonprot_3)']
)

# Save the comparison to a CSV file
merged_lengths.to_csv("sequence_length_comparison.csv", index=False)

# Summary of differences
print("\nSummary of Sequence Length Comparison:")
print(f"Total UniProt IDs compared: {len(merged_lengths)}")
print(f"Matching lengths: {merged_lengths['Length Difference'].isna().sum()}")
print(f"Non-matching lengths: {len(merged_lengths) - merged_lengths['Length Difference'].isna().sum()}")

# Save a unified UniProt IDs list to a CSV file
uniprot_ids_df = pd.DataFrame(all_uniprot_ids, columns=["UniProt IDs"])
uniprot_ids_df.to_csv("moonprot3_uniprot_ids_list.csv", index=False)
print("Unified UniProt IDs list saved to 'moonprot3_uniprot_ids_list.csv'.")


Summary of Sequence Length Comparison:
Total UniProt IDs compared: 513
Matching lengths: 268
Non-matching lengths: 245
Unified UniProt IDs list saved to 'moonprot3_uniprot_ids_list.csv'.


In [20]:
import re

# Split the 'go_terms' column into lists of GO annotations and extract only the GO codes
filtered_moonprot_3 = filtered_moonprot_3.copy()  # Ensure we work on a copy
filtered_moonprot_3['go_terms_split'] = filtered_moonprot_3['go_terms'].str.split(r'\r\n').apply(
    lambda x: [re.match(r'(GO:\d+)', term).group(1) for term in x if re.match(r'(GO:\d+)', term)] if isinstance(x, list) else []
)

# Explode the GO terms into individual rows for easier comparison
filtered_go_exploded = filtered_moonprot_3.explode('go_terms_split')[['uniprot_id', 'go_terms_split']].dropna()
filtered_go_exploded.rename(columns={'go_terms_split': 'GO Annotation'}, inplace=True)

# Display the first few rows to verify
print(filtered_go_exploded.head())

# Explode the GO terms into individual rows for easier comparison
filtered_go_exploded = filtered_moonprot_3.explode('go_terms_split')[['uniprot_id', 'go_terms_split']].dropna()
filtered_go_exploded.rename(columns={'go_terms_split': 'GO Annotation'}, inplace=True)

# Display the first few rows to verify
print(filtered_go_exploded.head())

  uniprot_id GO Annotation
0     Q9CW03    GO:0006275
0     Q9CW03    GO:0006281
0     Q9CW03    GO:0006974
0     Q9CW03    GO:0007049
0     Q9CW03    GO:0007052
  uniprot_id GO Annotation
0     Q9CW03    GO:0006275
0     Q9CW03    GO:0006281
0     Q9CW03    GO:0006974
0     Q9CW03    GO:0007049
0     Q9CW03    GO:0007052


In [17]:
moonprot_3_go.head(10)

,UniProt IDs,GO Annotation,GO Evidence,GoTerm,GO Category
0,P0AFW0,GO:0005829,IBA,C:cytosol,CC
1,P0AFW0,GO:0001000,IDA,F:bacterial-type RNA polymerase core enzyme bi...,MF
2,P0AFW0,GO:0003677,IEA,F:DNA binding,MF
3,P0AFW0,GO:0061980,IDA,F:regulatory RNA binding,MF
4,P0AFW0,GO:0001073,IDA,F:transcription antitermination factor activit...,MF
5,P0AFW0,GO:0008494,IMP,F:translation activator activity,MF
6,P0AFW0,GO:0006354,IDA,P:DNA-templated transcription elongation,BP
7,P0AFW0,GO:0045727,IMP,P:positive regulation of translation,BP
8,P0AFW0,GO:0031564,IDA,P:transcription antitermination,BP
9,P0AFW0,GO:0140673,IEA,P:transcription elongation-coupled chromatin r...,BP


In [26]:
# Prepare the moonprot_3_go dataset for comparison
moonprot_3_go_comparison = moonprot_3_go[['UniProt IDs', 'GO Annotation']].dropna()

# Perform the comparison to find matches and mismatches
merged_go_comparison = pd.merge(
    filtered_go_exploded,
    moonprot_3_go_comparison,
    left_on=['uniprot_id', 'GO Annotation'],
    right_on=['UniProt IDs', 'GO Annotation'],
    how='outer',  # Use outer join to find mismatches
    indicator=True  # Add a column to indicate the source of each row
)

# Count pairs that exist only in one dataset
only_in_filtered = merged_go_comparison[merged_go_comparison['_merge'] == 'left_only']
only_in_moonprot_3_go = merged_go_comparison[merged_go_comparison['_merge'] == 'right_only']

# Count pairs that exist in both datasets
matching_pairs = merged_go_comparison[merged_go_comparison['_merge'] == 'both']

# Count the number of unique UniProt IDs with mismatched pairs
uniprot_ids_only_in_filtered = only_in_filtered['uniprot_id'].nunique()
uniprot_ids_only_in_moonprot_3_go = only_in_moonprot_3_go['UniProt IDs'].nunique()

# Print the results
print(f"Number of UniProt ID - GO term pairs only in filtered_moonprot_3: {len(only_in_filtered)}")
print(f"Number of UniProt ID - GO term pairs only in moonprot_3_go: {len(only_in_moonprot_3_go)}")
print(f"Number of matching UniProt ID - GO term pairs: {len(matching_pairs)}")
print(f"Number of UniProt IDs with pairs only in filtered_moonprot_3: {uniprot_ids_only_in_filtered}")
print(f"Number of UniProt IDs with pairs only in moonprot_3_go: {uniprot_ids_only_in_moonprot_3_go}")

# Save the mismatched and matching pairs to CSV files
only_in_filtered.to_csv("pairs_only_in_filtered_moonprot_3.csv", index=False)
only_in_moonprot_3_go.to_csv("pairs_only_in_moonprot_3_go.csv", index=False)
matching_pairs.to_csv("matching_pairs_moonprot_3.csv", index=False)

Number of UniProt ID - GO term pairs only in filtered_moonprot_3: 3667
Number of UniProt ID - GO term pairs only in moonprot_3_go: 4350
Number of matching UniProt ID - GO term pairs: 4075
Number of UniProt IDs with pairs only in filtered_moonprot_3: 434
Number of UniProt IDs with pairs only in moonprot_3_go: 422
